# Basic Netrun Example

This notebook demonstrates:
1. Loading a network configuration from JSON
2. Creating and starting a Net
3. Injecting data into the network
4. Running the network until all processing is complete
5. Retrieving results from output queues

**Tip:** You can visualize and edit the network configuration by running `netrun-ui` in this folder.

## Load the Network Configuration

In [1]:
import json
from pathlib import Path

from netrun.core import Net, NetConfig

# Load the network configuration from JSON
config_path = Path("main.netrun.json")
config_data = json.loads(config_path.read_text())
config = NetConfig.model_validate(config_data)

print("Loaded config with nodes:")
for node in config.graph.nodes:
    print(f"  - {node.name}")

Loaded config with nodes:
  - double
  - add
  - format_result


## Create and Run the Network

The network flow is:
```
double(x=5) → 10 → add(a=10, b=10) → 20 → format_result(value=20) → "The answer is: 20"
```

In [2]:
async with Net(config) as net:
    # Inject input data:
    # - 'double' node receives x=5
    # - 'add' node receives b=10
    net.inject_data("double", "x", [5])
    net.inject_data("add", "b", [10])

    # Run until all processing is complete
    while True:
        # Move packets through edges
        await net.run_until_blocked()

        # Execute any startable epochs
        startable = net.get_startable_epochs()
        if not startable:
            break

        for epoch_id in startable:
            await net.execute_epoch(epoch_id)

    # Retrieve results from the output queue
    results = net.get_all_outputs("results")

    print("=" * 50)
    print("Results:")
    for packet in results:
        print(f"  {packet.value}")

    # Show captured print logs from all nodes
    print()
    print("Node Logs:")
    for node_name in ["double", "add", "format_result"]:
        logs = net.get_node_log(node_name)
        if logs:
            print(f"\n  [{node_name}]")
            for timestamp, message in logs:
                print(f"    {timestamp.strftime('%H:%M:%S.%f')[:-3]} | {message}", end="")

Results:
  The answer is: 20

Node Logs:

  [double]
    17:31:49.006 | Doubling 5
    17:31:49.006 | Result: 10

  [add]
    17:31:49.007 | Adding 10 + 10
    17:31:49.007 | Result: 20

  [format_result]
    17:31:49.008 | Formatting result: 20
    17:31:49.008 | Formatted: The answer is: 20


## Understanding the Node Functions

The node functions are defined in `nodes.py`. Let's look at them:

In [3]:
print(Path("nodes.py").read_text())

"""Node functions for the basic net example.

Each function becomes a node in the network. The function signature determines
the node's ports:
- Parameters become input ports (except 'ctx' and 'print' which are special)
- Return type becomes output port(s)

Special parameters:
- ctx: NodeExecutionContext - provides access to the execution context
- print: A captured print function that logs output with timestamps
"""


def double(x: int, print) -> int:
    """Double the input value."""
    print(f"Doubling {x}")
    result = x * 2
    print(f"Result: {result}")
    return result


def add(a: int, b: int, print) -> int:
    """Add two numbers together."""
    print(f"Adding {a} + {b}")
    result = a + b
    print(f"Result: {result}")
    return result


def format_result(value: int, print) -> str:
    """Format the result as a string and log it."""
    print(f"Formatting result: {value}")
    result = f"The answer is: {value}"
    print(f"Formatted: {result}")
    return result



## Understanding the Network Configuration

The network is defined in `main.netrun.json`:

In [4]:
print(json.dumps(config_data, indent=2))

{
  "meta": {
    "description": "A basic example demonstrating netrun's flow-based execution"
  },
  "output_queues": {
    "results": {
      "ports": [
        [
          "format_result",
          "out"
        ]
      ]
    }
  },
  "graph": {
    "nodes": [
      {
        "name": "double",
        "factory": "netrun.node_factories.function",
        "factory_args": {
          "func": "nodes.double"
        },
        "meta": {
          "ui": {
            "position": {
              "x": 300,
              "y": 100
            }
          }
        }
      },
      {
        "name": "add",
        "factory": "netrun.node_factories.function",
        "factory_args": {
          "func": "nodes.add"
        },
        "meta": {
          "ui": {
            "position": {
              "x": 500,
              "y": 150
            }
          }
        }
      },
      {
        "name": "format_result",
        "factory": "netrun.node_factories.function",
        "factory_args": {